## 1) Environment setup (Colab)

In [5]:
import sys, tensorflow as tf
print("Python  :", sys.version)
print("TF      :", tf.__version__)
print("CUDA ok :", tf.config.list_physical_devices('GPU'))

Python  : 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
TF      : 2.19.0
CUDA ok : []


## 2) Mount Google Drive and download the dataset

In [4]:
from google.colab import files
files.upload()

!unzip -qo DL_Assignment1_Dataset.zip -d /content/data

Saving DL_Assignment1_Dataset.zip to DL_Assignment1_Dataset.zip


In [6]:
from pathlib import Path
DATASET_ROOT = Path("/content/data/Dataset/Dataset")
print("DATASET_ROOT:", DATASET_ROOT)

DATASET_ROOT: /content/data/Dataset/Dataset


## 4) Build a unified pandas DataFrame

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

def build_dataframe_from_folder(ROOT: Path) -> pd.DataFrame:
    """
    Build a dataframe with columns: filename, expression, valence, arousal
    for datasets where each image has matching *_exp.npy, *_val.npy, *_aro.npy files.
    """
    images_dir = ROOT / "images"
    ann_dir    = ROOT / "annotations"

    rows = []
    exts = {".jpg", ".jpeg", ".png"}

    def load_scalar(path):
        arr = np.load(path, allow_pickle=True)
        if isinstance(arr, np.ndarray):
            return float(arr.flatten()[0])
        return float(arr)

    # Loop through image files
    for img_path in images_dir.iterdir():
        if not img_path.is_file() or img_path.suffix.lower() not in exts:
            continue

        base = img_path.stem
        exp_path = ann_dir / f"{base}_exp.npy"
        val_path = ann_dir / f"{base}_val.npy"
        aro_path = ann_dir / f"{base}_aro.npy"

        if not (exp_path.exists() and val_path.exists() and aro_path.exists()):
            continue

        try:
            exp = int(load_scalar(exp_path))
            val = load_scalar(val_path)
            aro = load_scalar(aro_path)

            rows.append({
                "filename": img_path.name,
                "expression": exp,
                "valence": val,
                "arousal": aro,
            })
        except Exception as e:
            print(f"Skipping {img_path.name} due to error: {e}")

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError("No samples found. Double-check folder structure and file naming.")

    if df["expression"].min() == 1 and df["expression"].max() == 8:
        df["expression"] = df["expression"] - 1

    return df

DATASET_ROOT = Path("/content/data/Dataset/Dataset")
df_all = build_dataframe_from_folder(DATASET_ROOT)

print(df_all.head())
print("Total samples:", len(df_all))


   filename  expression   valence   arousal
0   599.jpg           6 -0.269841  0.571429
1  2175.jpg           6 -0.380952  0.896825
2  4414.jpg           2 -0.738784 -0.264125
3  2878.jpg           5 -0.612672  0.478183
4  4842.jpg           6 -0.150794  0.349206
Total samples: 3999


In [8]:
print(df_all.shape)
print(df_all['expression'].value_counts(dropna=False).sort_index())
print(df_all[['valence','arousal']].describe())

(3999, 4)
expression
0    500
1    500
2    500
3    500
4    500
5    500
6    500
7    499
Name: count, dtype: int64
           valence      arousal
count  3999.000000  3999.000000
mean     -0.191865     0.353902
std       0.466286     0.379546
min      -0.987224    -0.666667
25%      -0.611111     0.015873
50%      -0.182540     0.440379
75%       0.031746     0.666667
max       0.982385     0.984127


## 5) Split into Train / Val / Test

In [9]:
from sklearn.model_selection import train_test_split
df_trainval, df_test = train_test_split(df_all, test_size=0.15, random_state=42, stratify=df_all['expression'])
df_train, df_val = train_test_split(df_trainval, test_size=0.1765, random_state=42, stratify=df_trainval['expression'])
print("train", df_train.shape, "val", df_val.shape, "test", df_test.shape)

train (2799, 4) val (600, 4) test (600, 4)


## 6) TF data pipeline (with augmentation) + per-output sample-weights

In [10]:
import tensorflow as tf
IMG_SIZE = (224, 224)
NUM_CLASSES = 8
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomContrast(0.1),
], name="augment")

def one_hot(y, num_classes=NUM_CLASSES):
    return tf.one_hot(tf.cast(y, tf.int32), num_classes)

def load_and_preprocess_img(img_path):
    img_raw = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def make_ds(df, img_root: Path, augment_on=False, shuffle=False):
    img_root = tf.constant(str(img_root / "images"))
    fns  = tf.constant(df["filename"].astype(str).values)
    expr = tf.constant(df["expression"].astype(int).values)
    val  = tf.constant(df["valence"].astype(float).values)
    aro  = tf.constant(df["arousal"].astype(float).values)

    ds = tf.data.Dataset.from_tensor_slices((fns, expr, val, aro))

    def _map(fn, e, v, a):
        path = tf.strings.join([img_root, fn], separator="/")
        x = load_and_preprocess_img(path)
        if augment_on:
            x = augment(x, training=True)
        y_expr = one_hot(e)
        y_val  = tf.expand_dims(v, -1)
        y_aro  = tf.expand_dims(a, -1)

        w_expr = tf.ones_like(y_val, dtype=tf.float32)
        w_val  = tf.where(tf.equal(v, -2.0), 0.0, 1.0)
        w_aro  = tf.where(tf.equal(a, -2.0), 0.0, 1.0)
        w_val  = tf.expand_dims(w_val, -1)
        w_aro  = tf.expand_dims(w_aro, -1)

        targets = {"expr": y_expr, "valence": y_val, "arousal": y_aro}
        weights = {"expr": w_expr, "valence": w_val, "arousal": w_aro}
        return x, targets, weights

    if shuffle:
        ds = ds.shuffle(len(df), reshuffle_each_iteration=True)
    ds = ds.map(_map, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(df_train, DATASET_ROOT, augment_on=True,  shuffle=True)
val_ds   = make_ds(df_val,   DATASET_ROOT, augment_on=False, shuffle=False)
test_ds  = make_ds(df_test,  DATASET_ROOT, augment_on=False, shuffle=False)
train_ds, val_ds, test_ds

(<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), {'expr': TensorSpec(shape=(None, 8), dtype=tf.float32, name=None), 'valence': TensorSpec(shape=(None, 1), dtype=tf.float64, name=None), 'arousal': TensorSpec(shape=(None, 1), dtype=tf.float64, name=None)}, {'expr': TensorSpec(shape=(None, 1), dtype=tf.float32, name=None), 'valence': TensorSpec(shape=(None, 1), dtype=tf.float32, name=None), 'arousal': TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)})>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), {'expr': TensorSpec(shape=(None, 8), dtype=tf.float32, name=None), 'valence': TensorSpec(shape=(None, 1), dtype=tf.float64, name=None), 'arousal': TensorSpec(shape=(None, 1), dtype=tf.float64, name=None)}, {'expr': TensorSpec(shape=(None, 1), dtype=tf.float32, name=None), 'valence': TensorSpec(shape=(None, 1), dtype=tf.float32, name=None), 'arousal': TensorSpec(shape=(None, 1), dty

## 7) Build models (VGG16, ResNet50) with shared heads

In [11]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pp
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pp
import tensorflow as tf

def build_multitask_model(backbone="VGG16", input_shape=(224,224,3), train_base=False):
    inp = layers.Input(shape=input_shape, name="image")
    if backbone=="VGG16":
        x = layers.Lambda(vgg_pp, name="preprocess")(inp)
        base = VGG16(weights="imagenet", include_top=False, input_tensor=x)
    elif backbone=="ResNet50":
        x = layers.Lambda(resnet_pp, name="preprocess")(inp)
        base = ResNet50(weights="imagenet", include_top=False, input_tensor=x)
    else:
        raise ValueError("Unsupported backbone. Use 'VGG16' or 'ResNet50'.")

    base.trainable = train_base
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.3)(x)

    expr = layers.Dense(256, activation="relu")(x)
    expr = layers.Dropout(0.3)(expr)
    expr = layers.Dense(NUM_CLASSES, activation="softmax", name="expr")(expr)

    val = layers.Dense(128, activation="relu")(x)
    val = layers.Dropout(0.2)(val)
    val = layers.Dense(1, activation="tanh", name="valence")(val)

    aro = layers.Dense(128, activation="relu")(x)
    aro = layers.Dropout(0.2)(aro)
    aro = layers.Dense(1, activation="tanh", name="arousal")(aro)

    model = models.Model(inputs=inp, outputs=[expr, val, aro], name=f"MTL_{backbone}")
    return model

def compile_model(model, lr=1e-3):
    losses = {"expr": "categorical_crossentropy","valence": "mse","arousal": "mse"}
    metrics = {
        "expr": ["accuracy"],
        "valence": [tf.keras.metrics.MeanAbsoluteError(name="mae"),
                    tf.keras.metrics.RootMeanSquaredError(name="rmse")],
        "arousal": [tf.keras.metrics.MeanAbsoluteError(name="mae"),
                    tf.keras.metrics.RootMeanSquaredError(name="rmse")]
    }
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=losses, metrics=metrics)
    return model

vgg = build_multitask_model("VGG16", train_base=False)
vgg = compile_model(vgg, lr=1e-3)
vgg.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "MTL_VGG16"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ preprocess (Lambda) │ (None, 224, 224,  │          0 │ image[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 224, 224,  │      1,792 │ preprocess[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 224, 224,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 112, 112,  │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 112, 112,  │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 112, 112,  │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 56, 56,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 56, 56,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 56, 56,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 56, 56,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 28, 28,    │          0 │ block3_conv3[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv1        │ (None, 28, 28,    │  1,180,160 │ block3_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv2        │ (None, 28, 28,    │  2,359,808 │ block4_conv1[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv3        │ (None, 28, 28,    │  2,359,808 │ block4_conv2[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_pool         │ (None, 14, 14,    │          0 │ block4_conv3[0][… │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv1        │ (None, 14, 14,    │  2,359,808 │ block4_pool[0][0

 Total params: 14,979,658 (57.14 MB)

 Trainable params: 264,970 (1.01 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

## 8) Train & fine-tune (VGG16 baseline)

In [ ]:
import numpy as np
import tensorflow as tf
from pathlib import Path

AUTOTUNE   = tf.data.AUTOTUNE
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomContrast(0.1),
], name="augment")

def load_and_preprocess_img(img_path):
    img_raw = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img

def one_hot(y, num_classes=NUM_CLASSES):
    return tf.one_hot(tf.cast(y, tf.int32), num_classes)

def make_ds(df, img_root: Path, augment_on=False, shuffle=False):
    img_root = tf.constant(str(img_root / "images"))

    fns  = tf.constant(df["filename"].astype(str).values)
    expr = tf.constant(df["expression"].astype(np.int32).values,  dtype=tf.int32)
    val  = tf.constant(df["valence"  ].astype(np.float32).values, dtype=tf.float32)
    aro  = tf.constant(df["arousal"  ].astype(np.float32).values, dtype=tf.float32)

    def _map(fn, e, v, a):
        path = tf.strings.join([img_root, fn], separator="/")
        x = load_and_preprocess_img(path)
        if augment_on:
            x = augment(x, training=True)

        y_expr = one_hot(e)
        y_val  = tf.expand_dims(v, -1)
        y_aro  = tf.expand_dims(a, -1)

        w_expr = tf.ones_like(e, dtype=tf.float32)
        w_val  = tf.where(tf.equal(v, -2.0), 0.0, 1.0)
        w_aro  = tf.where(tf.equal(a, -2.0), 0.0, 1.0)

        targets = {"expr": y_expr, "valence": y_val, "arousal": y_aro}
        weights = {"expr": w_expr, "valence": w_val, "arousal": w_aro}
        return x, targets, weights

    ds = tf.data.Dataset.from_tensor_slices((fns, expr, val, aro))
    if shuffle:
        ds = ds.shuffle(len(fns), reshuffle_each_iteration=True)
    ds = ds.map(_map, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

DATASET_ROOT = Path("/content/data/Dataset/Dataset")
train_ds = make_ds(df_train, DATASET_ROOT, augment_on=True,  shuffle=True)
val_ds   = make_ds(df_val,   DATASET_ROOT, augment_on=False, shuffle=False)
test_ds  = make_ds(df_test,  DATASET_ROOT, augment_on=False, shuffle=False)

def as_tuple_triplet(ds, autotu=tf.data.AUTOTUNE):
    def _map(x, y, w):
        y_tuple = (y["expr"], y["valence"], y["arousal"])
        w_tuple = (w["expr"], w["valence"], w["arousal"])
        return x, y_tuple, w_tuple
    return ds.map(_map, num_parallel_calls=autotu).prefetch(autotu)

train_ds_t = as_tuple_triplet(train_ds)
val_ds_t   = as_tuple_triplet(val_ds)

xb,(y1,y2,y3),(w1,w2,w3) = next(iter(train_ds_t))
print("x:", xb.shape, xb.dtype)
print("y:", y1.shape, y2.shape, y3.shape, "| dtypes:", y1.dtype, y2.dtype, y3.dtype)
print("w:", w1.shape, w2.shape, w3.shape, "| dtypes:", w1.dtype, w2.dtype, w3.dtype)

from pathlib import Path

ckpt_dir = Path("/content/outputs/checkpoints"); ckpt_dir.mkdir(parents=True, exist_ok=True)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(filepath=str(ckpt_dir / "best_VGG16.keras"),
                                       monitor="val_loss", save_best_only=True)
]

EPOCHS_HEADS = 8
EPOCHS_FT    = 12
HEAD_LR      = 1e-3
FT_LR        = 5e-5
UNFREEZE_N   = 30
vgg = build_multitask_model("VGG16", train_base=False)
vgg = compile_model(vgg, lr=HEAD_LR)

print("\n=== Training heads (frozen backbone) ===")
history_vgg_heads = vgg.fit(
    train_ds_t,
    validation_data=val_ds_t,
    epochs=EPOCHS_HEADS,
    callbacks=callbacks,
    verbose=1
)

# Phase 2: unfreeze top N layers and fine-tune at small LR
for layer in vgg.layers[-UNFREEZE_N:]:
    layer.trainable = True
vgg = compile_model(vgg, lr=FT_LR)

print("\n=== Fine-tuning top layers ===")
history_vgg_ft = vgg.fit(
    train_ds_t,
    validation_data=val_ds_t,
    epochs=EPOCHS_FT,
    callbacks=callbacks,
    verbose=1
)

# Optional quick validation accuracy on expression head
import numpy as np
def quick_acc(model, ds):
    accs = []
    for x, (y_expr, y_val, y_aro), _ in ds:
        y_true = tf.argmax(y_expr, axis=1).numpy()
        y_pred_list = model.predict(x, verbose=0)
        y_prob = y_pred_list[0]            # expr head
        y_hat  = y_prob.argmax(axis=1)
        accs.append((y_hat == y_true).mean())
    return float(np.mean(accs))

print(f"\nVal expr accuracy: {quick_acc(vgg, val_ds_t):.4f}")

# Save final model
out_dir = Path("/content/outputs"); out_dir.mkdir(parents=True, exist_ok=True)
vgg.save(out_dir / "final_VGG16.keras")
print("Saved:", out_dir / "final_VGG16.keras")

x: (32, 224, 224, 3) <dtype: 'float32'>
y: (32, 8) (32, 1) (32, 1) | dtypes: <dtype: 'float32'> <dtype: 'float32'> <dtype: 'float32'>
w: (32,) (32,) (32,) | dtypes: <dtype: 'float32'> <dtype: 'float32'> <dtype: 'float32'>

=== Training heads (frozen backbone) ===
Epoch 1/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 1783s 20s/step - arousal_loss: 0.5564 - arousal_mae: 0.6042 - arousal_rmse: 0.7341 - expr_accuracy: 0.1381 - expr_loss: 2.5409 - loss: 3.5920 - valence_loss: 0.4946 - valence_mae: 0.5637 - valence_rmse: 0.7029 - val_arousal_loss: 0.1457 - val_arousal_mae: 0.3230 - val_arousal_rmse: 0.3810 - val_expr_accuracy: 0.1250 - val_expr_loss: 2.0955 - val_loss: 2.4623 - val_valence_loss: 0.2216 - val_valence_mae: 0.3871 - val_valence_rmse: 0.4712 - learning_rate: 0.0010
Epoch 2/8
88/88 ━━━━━━━━━━━━━━━━━━━━ 1780s 20s/step - arousal_loss: 0.2695 - arousal_mae: 0.4142 - arousal_rmse: 0.5190 - expr_accuracy: 0.1360 - expr_loss: 2.1780 - loss: 2.8117 - valence_loss: 0.3641 - valence_mae: 0.4848 - valence_

## 9) Train & fine-tune (ResNet50 baseline)

In [ ]:
resnet = build_multitask_model("ResNet50", train_base=False)
resnet = compile_model(resnet, lr=1e-3)
callbacks2 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    ModelCheckpoint(filepath=str(ckpt_dir / "best_ResNet50.keras"),
                    monitor="val_loss", save_best_only=True)
]
history_res_heads = resnet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEADS, callbacks=callbacks2, verbose=1)

M = 50
for layer in resnet.layers[-M:]:
    layer.trainable = True
resnet = compile_model(resnet, lr=5e-5)
history_res_ft = resnet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, callbacks=callbacks2, verbose=1)

## 10) Training curves

In [ ]:
import matplotlib.pyplot as plt

def plot_history(h, title=""):
    hist = h.history
    plt.figure()
    plt.plot(hist["loss"], label="train")
    plt.plot(hist["val_loss"], label="val")
    plt.title(title or "Total Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)

plot_history(history_vgg_heads, "VGG16 (heads)")
plot_history(history_vgg_ft,    "VGG16 (fine-tune)")
plot_history(history_res_heads, "ResNet50 (heads)")
plot_history(history_res_ft,    "ResNet50 (fine-tune)")

## 11) Evaluation (All required metrics)

In [ ]:
from sklearn.metrics import f1_score, cohen_kappa_score, roc_auc_score, average_precision_score
import numpy as np, pandas as pd

def pearsonr_np(y_true, y_pred):
    y_true = np.asarray(y_true).ravel(); y_pred = np.asarray(y_pred).ravel()
    vt = y_true - y_true.mean(); vp = y_pred - y_pred.mean()
    denom = (np.sqrt((vt**2).sum()) * np.sqrt((vp**2).sum()) + 1e-12)
    return float((vt * vp).sum() / denom)

def rmse_np(y_true, y_pred):
    y_true = np.asarray(y_true).ravel(); y_pred = np.asarray(y_pred).ravel()
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))

def sagr_np(y_true, y_pred):
    y_true = np.asarray(y_true).ravel(); y_pred = np.asarray(y_pred).ravel()
    def sgn(x): return 1 if x >= 0 else -1
    signs_true = np.vectorize(sgn)(y_true); signs_pred = np.vectorize(sgn)(y_pred)
    return float(np.mean(signs_true == signs_pred))

def ccc_np(y_true, y_pred):
    y_true = np.asarray(y_true).ravel(); y_pred = np.asarray(y_pred).ravel()
    mu_x, mu_y = y_true.mean(), y_pred.mean()
    vx, vy = y_true.var(), y_pred.var()
    cov = np.mean((y_true - mu_x) * (y_pred - mu_y))
    return float((2 * cov) / (vx + vy + (mu_x - mu_y)**2 + 1e-12))

def krippendorff_alpha_nominal_two_raters(y1, y2):
    # Nominal alpha for exactly two raters
    y1 = np.asarray(y1).ravel(); y2 = np.asarray(y2).ravel()
    assert y1.shape == y2.shape
    N = len(y1)
    Do = np.mean(y1 != y2)
    all_ratings = np.concatenate([y1, y2], axis=0)
    _, counts = np.unique(all_ratings, return_counts=True)  # counts over combined ratings
    N_tot = counts.sum()  # equals 2N
    De = np.sum(counts * (N_tot - counts)) / (N_tot * (N_tot - 1) + 1e-12)
    if De == 0:
        return 1.0 if Do == 0 else 0.0
    return 1.0 - (Do / De)

def evaluate_model(model, dataset, split_name="val"):
    y_true_cls, y_prob_cls = [], []
    y_true_val, y_pred_val = [], []
    y_true_aro, y_pred_aro = [], []
    for batch in dataset:
        x, targets, _ = batch
        y_expr_true = np.argmax(targets["expr"].numpy(), axis=1)
        y_val_true  = targets["valence"].numpy().ravel()
        y_aro_true  = targets["arousal"].numpy().ravel()
        y_expr_prob, y_val_pred, y_aro_pred = model.predict(x, verbose=0)
        y_true_cls.append(y_expr_true); y_prob_cls.append(y_expr_prob)
        y_true_val.append(y_val_true);  y_pred_val.append(y_val_pred.ravel())
        y_true_aro.append(y_aro_true);  y_pred_aro.append(y_aro_pred.ravel())
    y_true_cls = np.concatenate(y_true_cls, axis=0)
    y_prob_cls = np.concatenate(y_prob_cls, axis=0)
    y_pred_cls = y_prob_cls.argmax(axis=1)
    y_true_val = np.concatenate(y_true_val, axis=0); y_pred_val = np.concatenate(y_pred_val, axis=0)
    y_true_aro = np.concatenate(y_true_aro, axis=0); y_pred_aro = np.concatenate(y_pred_aro, axis=0)

    acc = float((y_pred_cls == y_true_cls).mean())
    f1_macro = float(f1_score(y_true_cls, y_pred_cls, average="macro"))
    kappa = float(cohen_kappa_score(y_true_cls, y_pred_cls))
    alpha = float(krippendorff_alpha_nominal_two_raters(y_true_cls, y_pred_cls))

    y_true_onehot = np.eye(y_prob_cls.shape[1])[y_true_cls]
    try:
        auc_macro = float(roc_auc_score(y_true_onehot, y_prob_cls, multi_class="ovr", average="macro"))
    except Exception:
        auc_macro = float("nan")
    try:
        pr_auc_macro = float(average_precision_score(y_true_onehot, y_prob_cls, average="macro"))
    except Exception:
        pr_auc_macro = float("nan")

    mask_val = y_true_val != -2; mask_aro = y_true_aro != -2
    val_rmse = rmse_np(y_true_val[mask_val], y_pred_val[mask_val]) if mask_val.any() else float("nan")
    val_corr = pearsonr_np(y_true_val[mask_val], y_pred_val[mask_val]) if mask_val.any() else float("nan")
    val_sagr = sagr_np(y_true_val[mask_val], y_pred_val[mask_val]) if mask_val.any() else float("nan")
    val_ccc  = ccc_np(y_true_val[mask_val], y_pred_val[mask_val]) if mask_val.any() else float("nan")

    aro_rmse = rmse_np(y_true_aro[mask_aro], y_pred_aro[mask_aro]) if mask_aro.any() else float("nan")
    aro_corr = pearsonr_np(y_true_aro[mask_aro], y_pred_aro[mask_aro]) if mask_aro.any() else float("nan")
    aro_sagr = sagr_np(y_true_aro[mask_aro], y_pred_aro[mask_aro]) if mask_aro.any() else float("nan")
    aro_ccc  = ccc_np(y_true_aro[mask_aro], y_pred_aro[mask_aro]) if mask_aro.any() else float("nan")

    results = dict(split=split_name, acc=acc, f1_macro=f1_macro, kappa=kappa, alpha=alpha,
                   auc_macro=auc_macro, pr_auc_macro=pr_auc_macro,
                   val_rmse=val_rmse, val_corr=val_corr, val_sagr=val_sagr, val_ccc=val_ccc,
                   aro_rmse=aro_rmse, aro_corr=aro_corr, aro_sagr=aro_sagr, aro_ccc=aro_ccc)
    return results, (y_true_cls, y_pred_cls, y_prob_cls)

In [ ]:
from pathlib import Path
out_dir = Path("/content/outputs"); out_dir.mkdir(parents=True, exist_ok=True)

results = []
res_vgg_val, (y_true_vgg, y_pred_vgg, y_prob_vgg) = evaluate_model(vgg, val_ds, split_name="val (VGG16)")
res_vgg_val["model"] = "VGG16"; results.append(res_vgg_val)

res_res_val, (y_true_res, y_pred_res, y_prob_res) = evaluate_model(resnet, val_ds, split_name="val (ResNet50)")
res_res_val["model"] = "ResNet50"; results.append(res_res_val)

res_vgg_test, _ = evaluate_model(vgg, test_ds, split_name="test (VGG16)"); res_vgg_test["model"] = "VGG16"; results.append(res_vgg_test)
res_res_test, _ = evaluate_model(resnet, test_ds, split_name="test (ResNet50)"); res_res_test["model"] = "ResNet50"; results.append(res_res_test)

df_results = pd.DataFrame(results)
df_results.to_csv(out_dir / "results_summary.csv", index=False)
df_results

## 12) Save some correctly / incorrectly classified images

In [ ]:
import numpy as np
from PIL import Image
save_ok = Path("/content/outputs/correct"); save_bad = Path("/content/outputs/incorrect")
save_ok.mkdir(parents=True, exist_ok=True); save_bad.mkdir(parents=True, exist_ok=True)

def save_examples(df_ref, y_true, y_pred, prefix="VGG16", max_each=16):
    idxs = np.arange(len(y_true))
    ok = idxs[y_true==y_pred][:max_each]; bad = idxs[y_true!=y_pred][:max_each]
    for kind, subset in [("ok", ok), ("bad", bad)]:
        for i, idx in enumerate(subset):
            row = df_ref.iloc[idx]
            src = Path("/content/data/affect_dataset/images") / str(row["filename"])
            try:
                Image.open(src).convert("RGB").save((save_ok if kind=="ok" else save_bad) / f"{prefix}_{kind}_{i}_t{int(y_true[idx])}_p{int(y_pred[idx])}.jpg")
            except Exception:
                pass

save_examples(df_val.reset_index(drop=True), y_true_vgg, y_pred_vgg, prefix="VGG16")
save_examples(df_val.reset_index(drop=True), y_true_res, y_pred_res, prefix="ResNet50")
print("Saved:", save_ok, save_bad)

## 13) Confusion matrices

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

def plot_cm(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    plt.figure(); plt.imshow(cm, interpolation='nearest'); plt.title(title)
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.xticks(range(NUM_CLASSES)); plt.yticks(range(NUM_CLASSES))
    plt.colorbar(); plt.tight_layout()

plot_cm(y_true_vgg, y_pred_vgg, "VGG16 — Confusion Matrix (val)")
plot_cm(y_true_res, y_pred_res, "ResNet50 — Confusion Matrix (val)")

## 14) Save final models

In [ ]:
vgg.save("/content/outputs/final_VGG16.keras")
resnet.save("/content/outputs/final_ResNet50.keras")
print("Saved to /content/outputs/")

## 15) Utility: record experiment config

In [ ]:
import json, datetime, pathlib
config = {"img_size": [224,224], "batch_size": 32, "backbones": ["VGG16","ResNet50"],
          "epochs_heads": 8, "epochs_finetune": 12, "timestamp": datetime.datetime.utcnow().isoformat()+"Z"}
pathlib.Path("/content/outputs").mkdir(parents=True, exist_ok=True)
with open("/content/outputs/run_config.json","w") as f: json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

---
### Next steps for your Report
- Paste `results_summary.csv` into your PDF and add analysis.  
- Include **graphs**, **confusion matrices**, and **example images**.  
- Briefly justify **baseline choice** and **transfer learning strategy**.  
- Discuss **RMSE, CORR, SAGR, CCC** and which metric suits in-the-wild use.